In [ ]:
import os
import numpy as np
import scipy as sp
import matplotlib as mpl
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

In [ ]:
df_path= '../Fixed_Image_Data'
dfs_to_load = [f for f in os.listdir(df_path) if not f.startswith('.')]

dfs_to_load

In [ ]:
final_df=pd.DataFrame(columns=['label', 'area', 'centroid-0', 'centroid-1', 'mean_intensity',
       'solidity', 'eccentricity', 'median_intensity', 'Channel', 'Media', 'Time', 'Date', 'Strain', 'Replicate', 'Position', 'Plasmid'])

In [ ]:
for df in dfs_to_load:
    data = pd.read_csv(os.path.join(df_path, df), index_col=0)
    final_df= pd.concat([final_df, data], ignore_index=True)

In [ ]:
#Change the NaN values as necessary to pivot table  
replacement_value = 'No_pLL'
final_df['Plasmid'].fillna(replacement_value, inplace=True)
replacement_value2 = 'No_Strain'
final_df['Strain'].fillna(replacement_value2, inplace=True)
replacement_value3 = 'No_Time'
final_df['Time'].fillna(replacement_value3, inplace=True)
replacement_value4 = 'No_Media'
final_df['Media'].fillna(replacement_value4, inplace=True)
final_df

In [ ]:
#Use to check that all columns outside of mean_intensity and median_intensity are the same for each labelxposition combo
#Everything except median intensity and median intensity should print equal here, if not the pivot table will fail
column_check = final_df.groupby(['label', 'Position'])['area'].nunique()

# If all values are 1, it means 'Area' is the same for all combinations
if (column_check == 1).all():
    print("EQUAL")
else:
    print("UNEQUAL")

In [ ]:
# Pivot the DataFrame
pivot_final_df = final_df.pivot_table(index=['label', 'Position', 'area', 'centroid-0', 'centroid-1', 'solidity', 'eccentricity', 'Media', 'Time', 'Date', 'Replicate', 'Strain', 'Plasmid' ], columns='Channel', values='median_intensity').reset_index()

# Rename columns
pivot_final_df.columns.name = None  # remove the name of the columns axis
pivot_final_df = pivot_final_df.rename(columns={'GFP': 'GFP_median_intensity', 'Phase': 'Phase_median_intensity', 'TRITC': 'TRITC_median_Intensity'})
pivot_final_df

In [ ]:
#Plot and Print Mean and SD for eccentricity
#Note that pLL565 is PssaG-sfGFP(LVA)-mRuby2
#pLL749 is PssaB-sfGFP-mRuby2
#pLL889 is the promoterless sfGFP(-) control
plasmidtoplot = ['pLL565', 'pLL749', 'pLL889', 'No_pLL']
timetoplot = [0, 4]
straintoplot = ['WT']
mediatoplot = ['MGM', 'M9']
replicatetoplot = [1, 2, 3]
    

df_plot = pivot_final_df[pivot_final_df['Media'].isin(mediatoplot)&
                pivot_final_df['Plasmid'].isin(plasmidtoplot) & pivot_final_df['Strain'].isin(straintoplot)
                  & pivot_final_df['Time'].isin(timetoplot) & pivot_final_df['Replicate'].isin(replicatetoplot)] 

#plt.figure(figsize=(5,2.5))
ax = sns.histplot(data=df_plot, x='eccentricity', log_scale=False, stat='probability', common_norm=False, bins = 30, kde = True, hue = 'Replicate', legend = True)
#ax.set_xlim(left=0, right=45000)
#ax.set_ylim(top=1000, bottom=0)
plt.ylabel('count')
plt.xticks(rotation = 45)
plt.show()

ecc_std= df_plot['eccentricity'].std()
ecc_mean=df_plot['eccentricity'].mean()


In [ ]:
plasmidtoplot = [ 'pLL749', 'pLL565', 'pLL889', 'No_pLL']
timetoplot = [0, 4]
straintoplot = ['WT', 'KO'] #KO refers to ssrB KO used in the supplement
mediatoplot = ['MGM', 'M9']
replicatetoplot = [1, 2, 3]
    

df_plot = pivot_final_df[pivot_final_df['Media'].isin(mediatoplot)&
                pivot_final_df['Plasmid'].isin(plasmidtoplot) & pivot_final_df['Strain'].isin(straintoplot)
                  & pivot_final_df['Time'].isin(timetoplot) & pivot_final_df['Replicate'].isin(replicatetoplot)] 

#plt.figure(figsize=(5,2.5))
ax = sns.histplot(data=df_plot, x='solidity', log_scale=False, stat='probability', common_norm=False, bins = 30, kde = True, hue = 'Replicate', legend = True)
#ax.set_xlim(left=0, right=45000)
#ax.set_ylim(top=1000, bottom=0)
plt.ylabel('count')
plt.xticks(rotation = 45)
plt.show()



sol_std= df_plot['solidity'].std()
sol_mean=df_plot['solidity'].mean()
print(sol_std)
print(sol_mean)

In [ ]:
plasmidtoplot = [ 'pLL749', 'pLL565', 'pLL889',  'No_pLL']
timetoplot = [0, 4]
straintoplot = ['WT', 'KO']
mediatoplot = ['MGM', 'M9']
replicatetoplot = [1, 2, 3]
#postoplot =['20240115_094']
    

df_plot = pivot_final_df[pivot_final_df['Media'].isin(mediatoplot)&
                pivot_final_df['Plasmid'].isin(plasmidtoplot) & pivot_final_df['Strain'].isin(straintoplot)
                  & pivot_final_df['Time'].isin(timetoplot) & pivot_final_df['Replicate'].isin(replicatetoplot)] 

#plt.figure(figsize=(5,2.5))
ax = sns.histplot(data=df_plot, x='area', log_scale=False, stat='probability', common_norm=False, bins = 200, kde = True, hue = 'Replicate', legend = True)
ax.set_xlim(left=0, right=600)
#ax.set_ylim(top=1000, bottom=0)
plt.ylabel('count')
plt.xticks(rotation = 45)
ax.set_xlim(left=0, right=350)
#Here is where you can plot different lines for the values
#Area I think will be the only parameter that needs to be hard coded as data is skewed so an std based filter doesn't make sense
vertical_lines = [50, 250] 
for line in vertical_lines:
    plt.axvline(x=line, color='r', linestyle='--')


area_std= df_plot['area'].std()
area_mean=df_plot['area'].mean()
print(area_std)
print(area_mean)

In [ ]:
#Defining TRITC filters
plasmidtoplot = [ 'pLL749', 'pLL565', 'pLL889', 'No_pLL']
timetoplot = [0, 4]
straintoplot = ['WT', 'KO']
mediatoplot = ['MGM', 'M9']
replicatetoplot = [1, 2, 3]

tritc_wt_std = df_plot[df_plot['Plasmid'] == 'No_pLL']['TRITC_median_Intensity'].std()
tritc_wt_mean = df_plot[df_plot['Plasmid'] == 'No_pLL']['TRITC_median_Intensity'].mean()
print(tritc_wt_mean)
print(tritc_wt_std)

df_plot = pivot_final_df[pivot_final_df['Media'].isin(mediatoplot)&
                pivot_final_df['Plasmid'].isin(plasmidtoplot) & pivot_final_df['Strain'].isin(straintoplot)
                  & pivot_final_df['Time'].isin(timetoplot) & pivot_final_df['Replicate'].isin(replicatetoplot)] 

#plt.figure(figsize=(5,2.5))
ax = sns.histplot(data=df_plot, x='TRITC_median_Intensity', log_scale=True, stat='probability', common_norm=False, bins = 50, kde = True, hue = 'Plasmid', legend = True)
vertical_lines = [tritc_wt_mean, 2*tritc_wt_mean ]
for line in vertical_lines:
    plt.axvline(x=line, color='r', linestyle='--')


#ax.set_xlim(left=0, right=600)
#ax.set_ylim(top=1000, bottom=0)
plt.ylabel('count')
plt.xticks(rotation = 45)
plt.show()


In [ ]:
#Remove the WT samples from the final df (they will be filtered out w/ tritc anyways but don't need them past here)
keep_plasmids = ['pLL889', 'pLL565', 'pLL749']
no_wt_final_df = pivot_final_df[pivot_final_df['Plasmid'].isin(keep_plasmids)]
no_wt_final_df

In [ ]:
#Filtering - Makes a new df for each step, and one for all the steps, prints number of labels removed, this is sequential, but final filter is additive
tritc_df = no_wt_final_df[(no_wt_final_df['TRITC_median_Intensity']>=2*tritc_wt_mean)]
tritc_lines_removed = len(no_wt_final_df) - len(tritc_df)

area_df = tritc_df[(tritc_df['area']>= 50) & (tritc_df['area']<= 250)]
area_lines_removed = len(tritc_df) - len(area_df)

sol_df = area_df[(area_df['solidity'] >= 0.8)]
unfiltered = len(no_wt_final_df)
sol_lines_removed = len(area_df) - len(sol_df)

ecc_df = sol_df[(sol_df['eccentricity'] >= 0.6)]
ecc_lines_removed = len(sol_df) - len(ecc_df)

filtered_df = no_wt_final_df[(no_wt_final_df['TRITC_median_Intensity']>=2*tritc_wt_mean) & (no_wt_final_df['area']>= 50) & (no_wt_final_df['area']<= 250) & (no_wt_final_df['eccentricity'] >= 0.6) & (no_wt_final_df['solidity'] >= 0.8)]

all_lines_removed_additive = len(no_wt_final_df) - len(filtered_df)
all_lines_removed_sequential = len(no_wt_final_df)-len(ecc_df)
removed_df = no_wt_final_df[~no_wt_final_df.index.isin(filtered_df.index)]
print("Unfiltered labels:", unfiltered)
print('Number of labels removed from tritc:', tritc_lines_removed)
print("Number of labels removed from area value:", area_lines_removed)
print("Number of labels removed from solidity value:", sol_lines_removed)
print("Number of labels removed from ecc value:", ecc_lines_removed)
print("Number of labels removed with additive filters:", all_lines_removed_additive)
print("Number of labels removed with sequential filters:", all_lines_removed_sequential)

In [ ]:
filtered_df

In [ ]:
#Define DF you want to downsample from
keepstrains = ['WT']
keepplasmid = ['pLL565', 'pLL749'] #the two reporters I'm plotting from
keeptime = [4]
dftosamp = filtered_df[filtered_df['Strain'].isin (keepstrains) & filtered_df['Plasmid'].isin (keepplasmid) & filtered_df['Time'].isin (keeptime)]
dftosamp

In [ ]:
groups = dftosamp.groupby(['Plasmid', 'Strain', 'Replicate', 'Time', 'Media']).size().reset_index(name='Count')
print(groups)

In [ ]:
# Define the number of samples per group
samples_per_group = 450

# Function to sample each group
def sample_group(group):
    return group.sample(samples_per_group)

# Apply sampling to each group and concatenate the results
sampled_df = dftosamp.groupby(['Plasmid', 'Strain', 'Replicate', 'Time', 'Media'], group_keys=False).apply(sample_group)

print(sampled_df)

In [ ]:
#Sanity check to make sure downsample worked as expected
groups = sampled_df.groupby(['Plasmid', 'Strain', 'Replicate', 'Time', 'Media']).size().reset_index(name='Count')
print(groups)

In [ ]:
#Make the final KDE plots
plasmidtoplot1 = ['pLL565']
timetoplot1 = [4]
straintoplot1 = ['WT']
mediatoplot1 = ['MGM']
replicatetoplot1 = [1, 2, 3]

df_plot1 = sampled_df[sampled_df['Media'].isin(mediatoplot1)&
               sampled_df['Plasmid'].isin(plasmidtoplot1) & sampled_df['Strain'].isin(straintoplot1)
                  & sampled_df['Time'].isin(timetoplot1) & sampled_df['Replicate'].isin(replicatetoplot1)] 


plasmidtoplot2 = ['pLL565']
timetoplot2 = [4]
straintoplot2 = ['WT']
mediatoplot2 = ['M9']
replicatetoplot2 = [1, 2, 3]

df_plot2 = sampled_df[sampled_df['Media'].isin(mediatoplot2)&
               sampled_df['Plasmid'].isin(plasmidtoplot2) & sampled_df['Strain'].isin(straintoplot2)
                  & sampled_df['Time'].isin(timetoplot2) & sampled_df['Replicate'].isin(replicatetoplot2)] 



plasmidtoplot3 = ['pLL749']
timetoplot3 = [4]
straintoplot3 = ['WT']
mediatoplot3 = ['MGM']
replicatetoplot3 = [1, 2, 3]

df_plot3 = sampled_df[sampled_df['Media'].isin(mediatoplot3)&
               sampled_df['Plasmid'].isin(plasmidtoplot3) & sampled_df['Strain'].isin(straintoplot3)
                  & sampled_df['Time'].isin(timetoplot3) & sampled_df['Replicate'].isin(replicatetoplot3)] 


plasmidtoplot4 = ['pLL749']
timetoplot4 = [4]
straintoplot4 = ['WT']
mediatoplot4 = ['M9']
replicatetoplot4 = [1, 2, 3]

df_plot4 = sampled_df[sampled_df['Media'].isin(mediatoplot4)&
               sampled_df['Plasmid'].isin(plasmidtoplot4) & sampled_df['Strain'].isin(straintoplot4)
                  & sampled_df['Time'].isin(timetoplot4) & sampled_df['Replicate'].isin(replicatetoplot4)] 



# Create subplots with shared x-axis
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, sharex=True, figsize=(6, 8))

# Plot data on each subplot with the same bin edges
sns.kdeplot(data=df_plot1, x='GFP_median_intensity', log_scale=True, legend=True, fill = True, color = 'limegreen', ax=ax1)
ax1.set_ylabel('Density')
ax1.set_title('PssaG MgM-MES', y=1.0, x=0.8, pad=-14)

sns.kdeplot(data=df_plot2, x='GFP_median_intensity', log_scale=True, legend=True, fill = True, color = 'grey', ax=ax2)
ax2.set_ylabel('Density')
ax2.set_title('PssaG M9/Glc/CAA', y=1.0, x=0.8, pad=-14)

sns.kdeplot(data=df_plot3, x='GFP_median_intensity', log_scale=True, legend=True, fill = True, color = 'green', ax=ax3)
ax3.set_ylabel('Density')
ax3.set_title('PssaB MgM-MES', y=1.0, x=0.8, pad=-14)

sns.kdeplot(data=df_plot4, x='GFP_median_intensity', log_scale=True, legend=True, fill = True, color = 'black', ax=ax4)
ax4.set_ylabel('Density')
ax4.set_title('PssaB M9/Glc/CAA', y=1.0, x=0.8, pad=-14)

#plt.tight_layout()





In [ ]:
#Make the final KDE plots
plasmidtoplot1 = ['pLL565']
timetoplot1 = [4]
straintoplot1 = ['WT']
mediatoplot1 = ['MGM']
replicatetoplot1 = [1, 2, 3]

df_plot1 = sampled_df[sampled_df['Media'].isin(mediatoplot1)&
               sampled_df['Plasmid'].isin(plasmidtoplot1) & sampled_df['Strain'].isin(straintoplot1)
                  & sampled_df['Time'].isin(timetoplot1) & sampled_df['Replicate'].isin(replicatetoplot1)] 


plasmidtoplot2 = ['pLL565']
timetoplot2 = [4]
straintoplot2 = ['WT']
mediatoplot2 = ['M9']
replicatetoplot2 = [1, 2, 3]

df_plot2 = sampled_df[sampled_df['Media'].isin(mediatoplot2)&
               sampled_df['Plasmid'].isin(plasmidtoplot2) & sampled_df['Strain'].isin(straintoplot2)
                  & sampled_df['Time'].isin(timetoplot2) & sampled_df['Replicate'].isin(replicatetoplot2)] 



plasmidtoplot3 = ['pLL749']
timetoplot3 = [4]
straintoplot3 = ['WT']
mediatoplot3 = ['MGM']
replicatetoplot3 = [1, 2, 3]

df_plot3 = sampled_df[sampled_df['Media'].isin(mediatoplot3)&
               sampled_df['Plasmid'].isin(plasmidtoplot3) & sampled_df['Strain'].isin(straintoplot3)
                  & sampled_df['Time'].isin(timetoplot3) & sampled_df['Replicate'].isin(replicatetoplot3)] 


plasmidtoplot4 = ['pLL749']
timetoplot4 = [4]
straintoplot4 = ['WT']
mediatoplot4 = ['M9']
replicatetoplot4 = [1, 2, 3]

df_plot4 = sampled_df[sampled_df['Media'].isin(mediatoplot4)&
               sampled_df['Plasmid'].isin(plasmidtoplot4) & sampled_df['Strain'].isin(straintoplot4)
                  & sampled_df['Time'].isin(timetoplot4) & sampled_df['Replicate'].isin(replicatetoplot4)] 



# Create subplots with shared x-axis
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, sharex=True, figsize=(6, 8))

# Plot data on each subplot with the same bin edges
sns.kdeplot(data=df_plot1, x='TRITC_median_Intensity', log_scale=True, legend=True, fill = True, color = 'limegreen', ax=ax1)
ax1.set_ylabel('Count')
ax1.set_title('PssaG MgM-MES', y=1.0, x=0.8, pad=-14)

sns.kdeplot(data=df_plot2, x='TRITC_median_Intensity', log_scale=True, legend=True, fill = True, color = 'grey', ax=ax2)
ax2.set_ylabel('Density')
ax2.set_title('PssaG M9/Glc/CAA', y=1.0, x=0.8, pad=-14)

sns.kdeplot(data=df_plot3, x='TRITC_median_Intensity', log_scale=True, legend=True, fill = True, color = 'green', ax=ax3)
ax3.set_ylabel('Density')
ax3.set_title('PssaB MgM-MES', y=1.0, x=0.8, pad=-14)

sns.kdeplot(data=df_plot4, x='TRITC_median_Intensity', log_scale=True, legend=True, fill = True, color = 'black', ax=ax4)
ax4.set_ylabel('Density')
ax4.set_title('PssaB M9/Glc/CAA', y=1.0, x=0.8, pad=-14)

#plt.tight_layout()


In [ ]:
plasmidtoplot1 = ['pLL749']
timetoplot1 = [4]
straintoplot1 = ['WT']
mediatoplot1 = ['MGM']
replicatetoplot1 = [1, 2, 3]


df_plot1 = sampled_df[sampled_df['Media'].isin(mediatoplot1)&
               sampled_df['Plasmid'].isin(plasmidtoplot1) & sampled_df['Strain'].isin(straintoplot1)
                  & sampled_df['Time'].isin(timetoplot1) & sampled_df['Replicate'].isin(replicatetoplot1)] 

sns.kdeplot(data=df_plot1, x="GFP_median_intensity", y="TRITC_median_Intensity", log_scale=True, color = 'green', fill=False)


In [ ]:
#Define DF you want to downsample from - repeat all steps above to plot supplemental figures showing ssrB KO intensities with each reporter
keepstrains = ['KO']
keepplasmid = ['pLL565', 'pLL749']
keeptime = [4]
dftosamp2 = filtered_df[filtered_df['Strain'].isin (keepstrains) & filtered_df['Plasmid'].isin (keepplasmid) & filtered_df['Time'].isin (keeptime)]
dftosamp2